# GF(8) generalized toric codes: evolutionary search on Kaggle

This notebook runs the fixed-dimension evolutionary search from `evolve_search.py` on a Kaggle NVIDIA GPU for `k = 12..19`. It is a candidate factory: results are sampled and marked `verified: false`; promising candidates still need a stronger recheck or Magma/BZ verification.

Kaggle setup:

1. Create a Kaggle Notebook.
2. Set **Accelerator** to a GPU such as T4, P100, or A100.
3. Run the first cell to clone the GitHub repo into `/kaggle/working/generalized-toric-gpu-search`.
4. Run the remaining cells. JSONL result files are written to `/kaggle/working` and can be downloaded from the notebook output.

The notebook output is intentionally compact: one progress stream per `k`, then a summary table with the best saved set for each dimension. Full candidate records stay in the JSONL files.

Note: this repository is currently GF(8), length 49. The often-mentioned `[36,19,12]` example is GF(7), length 36, and would need a separate GF(7) evaluation matrix and kernels.

In [ ]:
# Clone the repo, matching the existing Kaggle notebooks in this project.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/t-ravnushkin/generalized-toric-gpu-search.git"
REPO_DIR = Path("/kaggle/working/generalized-toric-gpu-search")
BRANCH = None  # e.g. "main" or a feature branch; None uses the repo default.

os.chdir("/kaggle/working")

if REPO_DIR.exists():
    print(f"Repo already exists: {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    cmd = ["git", "clone", REPO_URL, str(REPO_DIR)]
    if BRANCH is not None:
        cmd = ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)]
    subprocess.run(cmd, check=True)

if not (REPO_DIR / "evolve_search.py").exists():
    raise FileNotFoundError(
        "The cloned repo does not contain evolve_search.py. Push the latest "
        "evolution-search changes to GitHub, or set REPO_URL/BRANCH to a fork "
        "that contains them."
    )

sys.path.insert(0, str(REPO_DIR))
print(f"Using repo: {REPO_DIR}")

In [ ]:
# GPU sanity check. CuPy is normally preinstalled on Kaggle GPU images.
REQUIRE_CUDA = True

try:
    import cupy as cp
    n_gpu = cp.cuda.runtime.getDeviceCount()
    print(f"CuPy {cp.__version__}; CUDA devices: {n_gpu}")
    for i in range(n_gpu):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(
            f"Device {i}: {props['name'].decode()} "
            f"({props.get('multiProcessorCount', 0)} SMs, "
            f"{props['totalGlobalMem'] // 1024**3} GB VRAM)"
        )
except Exception as exc:
    if REQUIRE_CUDA:
        raise RuntimeError("Enable a Kaggle GPU accelerator before running the search.") from exc
    print(f"CUDA unavailable; evolve_search.py will fall back if possible: {exc}")

In [ ]:
# Choose the search range and runtime profile.
from codetables import bounds_for_n

K_VALUES = list(range(12, 20))
FALLBACK_TARGETS = {12: 28, 13: 27, 14: 26, 15: 24, 16: 24, 17: 23, 18: 21, 19: 21}

try:
    bounds = bounds_for_n(49, q=8, cache=True)
    TARGETS = {k: bounds[k] for k in K_VALUES}
except Exception as exc:
    print(f"Could not fetch codetables bounds; using bundled fallback: {exc}")
    TARGETS = {k: FALLBACK_TARGETS[k] for k in K_VALUES}

# Start with quick to test that the notebook and GPU are healthy.
# Switch to serious for an overnight-style run.
PRESET = "quick"  # "quick" or "serious"

if PRESET == "quick":
    GENERATIONS = 20
    POPULATION_SIZE = 300
    SAMPLE_COUNT = 50_000
    RECHECK_SAMPLE_COUNT = 500_000
    RECHECK_ROUNDS = 2
    LOG_EVERY = 10
elif PRESET == "serious":
    GENERATIONS = 200
    POPULATION_SIZE = 300
    SAMPLE_COUNT = 200_000
    RECHECK_SAMPLE_COUNT = 2_000_000
    RECHECK_ROUNDS = 3
    LOG_EVERY = 20
else:
    raise ValueError("PRESET must be 'quick' or 'serious'")

ELITE_COUNT = 30
PARENT_POOL = 200
MUTATION_RATE = 0.10
RECHECK_TOP = 8
RECHECK_EVERY = 10
SEED_BASE = 1
BATCH_SIZE = None  # None lets evolve_search choose from the GPU SM count.
SEED_SETS_BY_K = {}  # Optional: {k: [(indices...), ...]} for seeded restarts.

print(f"Preset: {PRESET}")
print("Targets:")
for k in K_VALUES:
    print(f"  GF(8) [49,{k},?>={TARGETS[k]}]")

In [ ]:
# Run the k=12..19 sweep. Candidates are saved to JSONL files in /kaggle/working.
from datetime import datetime
from pathlib import Path
from evolve_search import init_oracles, run_evolution

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
ORACLES_INFO = init_oracles()  # Compile/init CUDA once and reuse it for every k.
RESULT_FILES = {}

for k in K_VALUES:
    results_file = Path(f"/kaggle/working/evolve_k{k}_{PRESET}_{ts}.json")
    RESULT_FILES[k] = results_file
    print("\n" + "=" * 72)
    print(f"k={k} target_d={TARGETS[k]} -> {results_file}")
    print("=" * 72)
    run_evolution(
        k=k,
        target_d=TARGETS[k],
        generations=GENERATIONS,
        population_size=POPULATION_SIZE,
        elite_count=ELITE_COUNT,
        parent_pool=PARENT_POOL,
        mutation_rate=MUTATION_RATE,
        sample_count=SAMPLE_COUNT,
        recheck_sample_count=RECHECK_SAMPLE_COUNT,
        recheck_rounds=RECHECK_ROUNDS,
        recheck_top=RECHECK_TOP,
        recheck_every=RECHECK_EVERY,
        batch_size=BATCH_SIZE,
        seed=SEED_BASE + 10_000 * k,
        seed_sets=[tuple(s) for s in SEED_SETS_BY_K.get(k, [])],
        results_file=results_file,
        oracles_info=ORACLES_INFO,
        log_every=LOG_EVERY,
    )

print("\nSweep complete.")
for k, path in RESULT_FILES.items():
    print(f"k={k}: {path}")

In [ ]:
# Compact actual results: best saved set per k, plus download links.
import json
from IPython.display import FileLink, display

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def point_str(points):
    return "[" + ", ".join(f"({a},{b})" for a, b in points) + "]"

SUMMARY = []
for k in K_VALUES:
    path = RESULT_FILES[k]
    records = load_jsonl(path)
    candidates = [r for r in records if r.get("type") == "candidate"]
    complete = next((r for r in reversed(records) if r.get("type") == "run_complete"), {})
    if candidates:
        best = max(
            candidates,
            key=lambda r: (
                r.get("recheck_min_distance", -1),
                r.get("sampled_min_distance", -1),
                r.get("generation", -1),
            ),
        )
        source = "candidate"
        best_d = best.get("recheck_min_distance")
        sampled_d = best.get("sampled_min_distance")
        indices = best.get("indices")
        points = best.get("lattice_points")
        generation = best.get("generation")
    else:
        source = "sample-best"
        best_d = None
        sampled_d = complete.get("best_sampled_min_distance")
        indices = complete.get("best_indices")
        points = complete.get("best_lattice_points")
        generation = None
    SUMMARY.append({
        "k": k,
        "target": TARGETS[k],
        "candidates": len(candidates),
        "source": source,
        "best_recheck_d": best_d,
        "best_sampled_d": sampled_d,
        "generation": generation,
        "indices": indices,
        "points": points,
        "file": path,
    })

print("Best saved result per k")
print("k  target  cand  source       recheck  sampled  gen")
print("-" * 62)
for row in SUMMARY:
    recheck = "-" if row["best_recheck_d"] is None else str(row["best_recheck_d"])
    sampled = "-" if row["best_sampled_d"] is None else str(row["best_sampled_d"])
    gen = "-" if row["generation"] is None else str(row["generation"])
    print(
        f"{row['k']:2d} {row['target']:7d} {row['candidates']:5d} "
        f"{row['source']:<12s} {recheck:>7s} {sampled:>8s} {gen:>4s}"
    )

print("\nBest sets")
for row in SUMMARY:
    print(f"\nk={row['k']} file={row['file'].name}")
    print(f"indices: {row['indices']}")
    print(f"points : {point_str(row['points']) if row['points'] else None}")

print("\nDownloads")
for row in SUMMARY:
    display(FileLink(str(row["file"])))

## What to do with results

The summary above prints one best saved set per `k`. Rows marked `candidate` survived the stronger sampled recheck; rows marked `sample-best` are only the best population member seen by sampled scoring and did not produce a saved rechecked candidate. Either way, these are not certificates.

For finalists, increase `RECHECK_SAMPLE_COUNT` and `RECHECK_ROUNDS`, then export the best sets for Magma's `VerifyMinimumDistanceLowerBound` / `MinimumDistance` workflow. Full candidate records remain in each JSONL file.